In [ ]:
%load_ext autoreload
%autoreload 2

from edunn import utils
import edunn as nn
import numpy as np

# Capa de Error Cuadrático

En este ejercicio debés implementar la capa de error `SquaredError`, que permite calcular el error de un lote de ejemplos. 

Las capas de error son diferentes a las capas normales por dos motivos:

1. No solo tienen como entrada la salida de la capa anterior, sino también el valor esperado de la capa anterior (`y` e `y_true`). 
2. Para un lote de $n$ ejemplos, su salida es un vector de tamaño $n$. Es decir, indican el valor del error de cada ejemplo con un escalar (número real).

La capa de error también debe poder realizar la operación `backward` de modo de poder propagar hacia atrás el gradiente del error a la red. 



# Método forward


El método `forward` de la capa `SquaredError` simplemente debe calcular la distancia euclídea al cuadrado entre `y`, los valores producidos por la red, e `y_true`, el valor esperado por la misma.

Por ejemplo, si $y=[2,-2]$ y $y_{true}=[3,3]$, entonces la salida de la capa es:

$$E(y,y_{true})=d_2(y,y_{true})=d_2([2,-2],[3,3])=(2-3)^2+(-2-3)^2 = 1^2+(-5)^2=1+25=26$$

En general, dados dos vectores $a=[a_1,\dots,a_n]$ y $b=[b_1,\dots,b_n]$, la distancia euclídea al cuadrado $d_2$ es:

$$
d_2(a,b)= d_2([a_1,\dots,a_n],[b_1,\dots,b_n]) =(a_1-b_1)^2+\dots+(a_n-b_n)^2
$$

En el caso de un lote de ejemplos, el cálculo es independiente para cada ejemplo. Es importante entonces que la suma de las diferencias al cuadrado se haga por cada ejemplo (fila) y no por cada característica (columna).


In [ ]:
y = np.array([[2, -2], [-4, 4]])
y_true = np.array([[3, 3], [-5, 2]])


layer = nn.SquaredError()
E = np.array([[26], [5]])
utils.check_same(E, layer.forward(y, y_true))

# Método backward

Ahora puedes calcular el error de una red, bien! Esta es la capa final de la red cuando se está entrenando. Por ende el método backward de una capa de error no recibe $\frac{dE}{dy}$; de hecho, debe calcularlo directamente a partir de $y$, $y_{true}$, y la definición del error. Además tampoco hay parámetros.

Por ende, en este caso, la derivada es simple. Solo debemos calcular $\frac{dE}{dy}$, la derivada del error respecto a la salida calculada por la red, $y$.
En este caso $E$ es simétrico respecto de sus entradas, así que llamemosla nuevamente $a$ y $b$, y entonces calculemos la derivada respecto del elemento $i$ de $a$ (la de $b$ sería igual):

$$
\frac{dE(a,b)}{da_i} = \frac{d((a_1-b_1)^2+\dots+(a_n-b_n)^2)}{da_i} \\
= \frac{d((a_i-b_i)^2)}{da_i} = 2 (a_i-b_i) \frac{d((a_i-b_i))}{da_i} \\
= 2 (a_i-b_i) 1 = 2 (a_i-b_i)
$$
Generalizando para todo el vector $a$, entonces:
$$
\frac{dE(a,b)}{da} = 2 (a-b)
$$
Donde $a-b$ es una resta entre vectores.

Nuevamente, como este error es por cada ejemplo, entonces los cálculos son independientes en cada fila.

In [ ]:
# number of random values of x and dE_dy to generate and test gradients
samples = 100
batch_size = 2
features_in = 3
features_out = 5
input_shape = (batch_size, features_in)


layer = nn.SquaredError()
utils.check_gradient.squared_error(layer, input_shape, samples=samples)

<details>
<summary>Ayuda:</summary>

La descripción matemática vista para la capa de error `nn.SquaredError()` calcula el error para un batch de 1 ejemplos. Como generalmente tendremos algo del estilo: `nn.MeanError(nn.SquaredError())`, el método `backward` de la capa de error `SquaredError` recibe `dE_dEi`, calculado por la capa siguiente (en este caso `nn.MeanError()`).

### Caso por lotes

Ahora $y$ es una matriz de $N \times O$, siendo $N$ el tamaño del batch de ejemplos y $O$ el tamaño de features de salida.

$$
\begin{array}{c}
\hspace{30pt}\text{features} \\
y(x) = 
\begin{bmatrix}
y_{11} & y_{12} & \cdots & y_{1O} \\
y_{21} & y_{22} & \cdots & y_{2O} \\
\vdots & \vdots & \ddots & \vdots \\
y_{N1} & y_{N2} & \cdots & y_{NO}
\end{bmatrix}
\end{array}
\begin{array}{c}
%\\
%\\
%\\
\\
\text{samples}
\end{array}
$$

De modo que el error queda con el formato:

$$
E(y,y') = \begin{bmatrix}
\sum_{i=1}^{O} (y_{1i} - y'_{1i})^2 \\
\sum_{i=1}^{O} (y_{2i} - y'_{2i})^2 \\
\vdots \\
\sum_{i=1}^{O} (y_{Ni} - y'_{Ni})^2
\end{bmatrix}
= \begin{bmatrix}
E_1(y_{1,:}, y'_{1,:}) \\
E_2(y_{2,:}, y'_{2,:}) \\
\vdots \\
E_N(y_{N,:}, y'_{N,:})
\end{bmatrix}
$$

Sin embargo, si estamos calculando el error para un conjunto de ejemplos, deberá existir una capa más seguida de la capa `nn.SquaredError`, la cual realice el promedio de error de tales ejemplos. Es por ello que desde ahora llamaremos $E=E_{\text{Mean}}$ a la **última** capa de error, sin utilizar tal subíndice de modo de no contribuir a la contaminación visual de las expresiones matemáticas.

Se debe calcular $\frac{dE}{dy}$ (análogo a $\frac{dE}{dx}$ en las capa previas), reduciendo la notación de $E_i(y_{i,:}, y'_{i,:})$ por simplicidad a $E_i(y_i, y'_i)=\sum_{j=1}^{O} (y_{i,j} - y'_{i,j})^2$, podemos ver que $\frac{dE_i}{dy_i} = 2 (y_i-y'_i)$ (que coincide con la derivada calculada anteriormente).

Por lo tanto, podemos tratar a $E(E_1,E_2,...,E_N)$ en función de los errores individuales, así su derivada para $y_i$ se calcula utilizando la regla de la cadena para funciones de varias variables:

$$\frac{dE}{dy_i} = \frac{dE}{dE_1} * \frac{dE_1}{dy_i} + \frac{dE}{dE_2} * \frac{dE_2}{dy_i} + ... + \frac{dE}{dE_N} * \frac{dE_N}{dy_i} = \sum_{k=1}^{N} \frac{dE}{dE_k} * \frac{dE_k}{dy_i} $$

Y como solo $E_i$ depende de $y_i$, la expresión queda reducida al termino donde $k=i$:

$$\frac{dE}{dy_i} = \frac{dE}{dE_i} * \frac{dE_i}{dy_i}$$

Además, como estamos agrupando las columnas de $y$ por los ejemplos en el lote: $y \in \mathbb{R}^N$. Lo cual es compatible con que $E \in \mathbb{R}^N$ y en consecuencia con la derivada de la relación de ambos vectores columna: $\frac{dE}{dy} \in \mathbb{R}^N$.

$$\frac{dE}{dy} = \frac{dE}{dE_i} ⊙ \frac{dE_i}{dy}$$

**Importante**: se expresa $E_i$ para representar a $E_{\text{SquaredError}}(y,y')$, únicamente para que se respete el nombre de las variables con respecto a la implementación en la librería.

Esto no es más que el producto elemento a elemento ([element-wise product](https://en.wikipedia.org/wiki/Hadamard_product_(matrices))) de dos vectores. En `numpy`, se puede calcular simplemente con `a*b`.

<!-- $$
y = \left[\begin{array}{cccc}
y_{1,1} & y_{1,2} & \cdots & y_{1,O} \\
y_{2,1} & y_{2,2} & \cdots & y_{2,O} \\
\vdots  & \vdots  & \ddots & \vdots  \\
y_{N,1} & y_{N,2} & \cdots & y_{N,O}
\end{array}\right]
\begin{array}{c}
\left.\rule{0pt}{15pt}\right\}\text{samples}\\
\\
\\
\\
\end{array}
\begin{array}{c}
\\
\\
\\
\rule{0pt}{15pt}\underbrace{\hspace{80pt}}_\text{features}
\end{array}
$$ -->

<!-- Tener en cuenta que el cálculo de `dE_dy` es la misma idea que cuando calculamos `dE_dx` para las capa previas, solo que como esta es la capa de error, lo que "sale" de esta capa en el `forward` es `E` y no una `y`, de modo que lo que "entra" a esta capa en el `forward` es la `y` de la capa anterior pero internamente no tenemos que considerarla como `x` como hacíamos antes ya que no devolvemos un `y` en el `forward` (sino que, como ya se dijo, devolvemos un `E`). -->

</details>